# 03 — RMSD Analysis

Computes per-frame RMSD for the same multi-system / multi-replicate trajectory
layout used by `01_hbond_analysis.ipynb`, and reports the results as time
series, distributions, and a summary table.

### Two RMSD definitions
| Function | Reference | Use |
|---|---|---|
| `calculate_rmsd_to_average` | The replicate's own **average structure** | Structural spread within a replicate 
| `calculate_rmsd_to_reference` | A **single fixed frame** (default: frame 0 of replicate 1) | Conventional drift-from-start trace; lets replicates be compared on a common reference |

Both use exactly the same Kabsch/SVD superposition, so the two traces are
directly comparable in magnitude. Choose which one to run with
`RMSD_MODE` in the configuration cell.

### Outputs
| File | Contents |
|---|---|
| `<System>_rmsd_timeseries.csv` | Frame-by-frame RMSD, one row per frame per replicate |
| `<System>_rmsd_summary.csv` | Mean / SD / min / max per replicate |
| `<System>_rmsd_timeseries.png` | Time series, one panel per replicate plus overlay |
| `<System>_rmsd_distribution.png` | Histogram per replicate |

### Environment
Same as notebook 01: Google Colab, trajectories on Google Drive, MDAnalysis as
the only non-default dependency. No new packages are introduced.

## 1. Setup

In [ ]:
!pip install MDAnalysis -q

In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda

from google.colab import files, drive

warnings.filterwarnings('ignore')

In [ ]:
drive.mount('/content/drive')

---
## 2. Configuration — edit this section only

The system dictionaries use the **same format** as `01_hbond_analysis.ipynb`,
so a configuration block can be copied between the two notebooks unchanged.
The donor/acceptor selections are ignored here but are harmless if left in
place.

In [ ]:
# ============================================================================
# SIMULATION SET 1  (same path format as 01_hbond_analysis.ipynb)
# ============================================================================
system_1 = {
    'name': 'System_1',
    'pdb_file': 'INSERT_TOPOLOGY_FILENAME.pdb',
    'dcd_files': [
        'INSERT_TRAJECTORY_FILENAME_R1.dcd',
        'INSERT_TRAJECTORY_FILENAME_R2.dcd',
        'INSERT_TRAJECTORY_FILENAME_R3.dcd',
        # Add more replicates as needed:
        # 'INSERT_TRAJECTORY_FILENAME_R4.dcd',
    ],
}

# ============================================================================
# SIMULATION SET 2  (set `system_2 = None` if only one system is analyzed)
# ============================================================================
system_2 = None

# system_2 = {
#     'name': 'System_2',
#     'pdb_file': 'INSERT_TOPOLOGY_FILENAME.pdb',
#     'dcd_files': ['INSERT_TRAJECTORY_FILENAME_R1.dcd'],
# }

# ============================================================================
# RMSD PARAMETERS
# ============================================================================
# Atoms used for both the superposition and the RMSD. Examples:
#   'protein and name CA'          - backbone C-alpha of the whole complex
#   'segid A and name CA'          - a single chain
RMSD_SELECTION = 'protein and name CA'

# 'average'   -> RMSD of each frame to its own replicate's average structure
#                (identical to the function used in notebooks 01 and 02)
# 'reference' -> RMSD of each frame to one fixed reference frame
RMSD_MODE = 'average'

# ============================================================================
# EQUILIBRATION  (must match 01_hbond_analysis.ipynb)
# ============================================================================
#   EQUILIBRATION_NS = 100.0  ->  1,000 frames  (DEFAULT)
FRAME_INTERVAL_NS = 0.1
EQUILIBRATION_NS = 100.0

FRAMES_TO_SKIP = int(round(EQUILIBRATION_NS / FRAME_INTERVAL_NS))

# Only used when RMSD_MODE == 'reference'.
REFERENCE_REPLICATE_IDX = 0   # 0-based index into dcd_files
REFERENCE_FRAME_IDX = 0       # frame within that replicate

# ============================================================================
# COLLECT SYSTEMS
# ============================================================================
systems = [system_1]
if system_2 is not None:
    systems.append(system_2)

print(f"Will analyze {len(systems)} system(s)  [mode: {RMSD_MODE}]")
print(f"Equilibration discarded from statistics: {FRAMES_TO_SKIP} frames "
      f"({EQUILIBRATION_NS} ns) per replicate")
for i, sys_cfg in enumerate(systems, 1):
    print(f"  {i}. {sys_cfg['name']}: {len(sys_cfg['dcd_files'])} replicates")

## 3. Verify that every input file exists

In [ ]:
print("Checking files...")
print("=" * 60)
all_good = True

for sys_cfg in systems:
    print(f"\n{sys_cfg['name']}:")

    if os.path.exists(sys_cfg['pdb_file']):
        print(f"  [OK]      PDB: {sys_cfg['pdb_file']}")
    else:
        print(f"  [MISSING] PDB: {sys_cfg['pdb_file']}")
        all_good = False

    for dcd in sys_cfg['dcd_files']:
        if os.path.exists(dcd):
            print(f"  [OK]      DCD: {dcd}")
        else:
            print(f"  [MISSING] DCD: {dcd}")
            all_good = False

print("\n" + "=" * 60)
if all_good:
    print("All files found. Ready to proceed.")
else:
    print("WARNING: some files are missing - check the paths listed above.")

---
## 4. RMSD functions

`calculate_rmsd_to_average` is the same function used in
`01_hbond_analysis.ipynb`, so RMSD values are directly comparable between
notebooks. `calculate_rmsd_to_reference` applies the same centring and
Kabsch/SVD superposition against a fixed reference coordinate set.

In [ ]:
def calculate_rmsd_to_average(universe, selection="protein and name CA"):
    """Per-frame RMSD to the trajectory-average structure.

    The average positions are accumulated over the whole trajectory, then each
    frame is centred and optimally rotated onto the (centred) average structure
    with a Kabsch/SVD superposition before the RMSD is evaluated.

    Parameters
    ----------
    universe : MDAnalysis.Universe
        Universe built from the topology and a single replicate trajectory.
    selection : str
        MDAnalysis selection used for both the averaging and the fit.

    Returns
    -------
    numpy.ndarray
        Array of shape (n_frames,) with the RMSD (Angstrom) of every frame to
        the average structure.
    """
    sel = universe.select_atoms(selection)
    n_frames = len(universe.trajectory)

    # ---- Pass 1: accumulate the average positions ----------------------
    avg_pos = np.zeros((len(sel), 3))
    for ts in universe.trajectory:
        avg_pos += sel.positions
    avg_pos /= n_frames

    # ---- Pass 2: superimpose each frame onto the average ---------------
    rmsd_to_avg = np.zeros(n_frames)
    for i, ts in enumerate(universe.trajectory):
        positions = sel.positions - sel.positions.mean(axis=0)
        avg_centered = avg_pos - avg_pos.mean(axis=0)

        # Kabsch rotation from the SVD of the cross-correlation matrix.
        correlation_matrix = np.dot(positions.T, avg_centered)
        U, S, Vt = np.linalg.svd(correlation_matrix)
        rotation = np.dot(U, Vt)

        # Guard against an improper rotation (reflection).
        if np.linalg.det(rotation) < 0:
            Vt[-1, :] *= -1
            rotation = np.dot(U, Vt)

        aligned = np.dot(positions, rotation)
        rmsd_to_avg[i] = np.sqrt(np.mean(np.sum((aligned - avg_centered) ** 2, axis=1)))

    return rmsd_to_avg

In [ ]:
def get_reference_positions(pdb_file, dcd_file, frame_idx=0, selection="protein and name CA"):
    """Coordinates of `selection` at a single frame, used as an RMSD reference."""
    u_ref = mda.Universe(pdb_file, dcd_file)
    u_ref.trajectory[frame_idx]
    return u_ref.select_atoms(selection).positions.copy()


def calculate_rmsd_to_reference(universe, ref_positions, selection="protein and name CA"):
    """Per-frame RMSD to a fixed reference coordinate set.

    Uses the same centring and Kabsch/SVD superposition as
    `calculate_rmsd_to_average`, so values from the two functions are on the
    same scale.

    Parameters
    ----------
    universe : MDAnalysis.Universe
        Universe built from the topology and a single replicate trajectory.
    ref_positions : numpy.ndarray
        (n_atoms, 3) reference coordinates for the same `selection`, e.g. from
        `get_reference_positions`.
    selection : str
        MDAnalysis selection used for both the fit and the RMSD.

    Returns
    -------
    numpy.ndarray
        Array of shape (n_frames,) with the RMSD (Angstrom) of every frame to
        the reference.
    """
    sel = universe.select_atoms(selection)
    n_frames = len(universe.trajectory)

    if len(sel) != len(ref_positions):
        raise ValueError(
            f"Selection has {len(sel)} atoms but the reference has "
            f"{len(ref_positions)} - the reference must come from a system with "
            "an identical selection."
        )

    ref_centered = ref_positions - ref_positions.mean(axis=0)

    rmsd_to_ref = np.zeros(n_frames)
    for i, ts in enumerate(universe.trajectory):
        positions = sel.positions - sel.positions.mean(axis=0)

        correlation_matrix = np.dot(positions.T, ref_centered)
        U, S, Vt = np.linalg.svd(correlation_matrix)
        rotation = np.dot(U, Vt)

        if np.linalg.det(rotation) < 0:
            Vt[-1, :] *= -1
            rotation = np.dot(U, Vt)

        aligned = np.dot(positions, rotation)
        rmsd_to_ref[i] = np.sqrt(np.mean(np.sum((aligned - ref_centered) ** 2, axis=1)))

    return rmsd_to_ref

---
## 5. Run the analysis

Each replicate is loaded in turn and its RMSD trace computed with the function
selected by `RMSD_MODE`. Results are stored in `rmsd_results` with the same
`{system: {'config': ..., 'replicates': [...]}}` shape used by the H-bond
notebook.

In [ ]:
rmsd_results = {}

for sys_config in systems:
    sys_name = sys_config['name']
    print(f"\n{'='*80}\nRMSD: {sys_name}\n{'='*80}")

    sys_results = {'config': sys_config, 'replicates': [], 'mode': RMSD_MODE,
                   'selection': RMSD_SELECTION}

    # In 'reference' mode every replicate is compared to one common frame.
    ref_positions = None
    if RMSD_MODE == 'reference':
        ref_dcd = sys_config['dcd_files'][REFERENCE_REPLICATE_IDX]
        ref_positions = get_reference_positions(
            sys_config['pdb_file'], ref_dcd,
            frame_idx=REFERENCE_FRAME_IDX, selection=RMSD_SELECTION
        )
        print(f"Reference: {os.path.basename(ref_dcd)} frame {REFERENCE_FRAME_IDX} "
              f"({len(ref_positions)} atoms)")

    for rep_idx, dcd_file in enumerate(sys_config['dcd_files']):
        rep_name = f"Rep{rep_idx + 1}"
        print(f"\n--- {rep_name}: {os.path.basename(dcd_file)} ---")

        u = mda.Universe(sys_config['pdb_file'], dcd_file)
        n_frames = len(u.trajectory)
        print(f"Loaded {n_frames} frames")

        if RMSD_MODE == 'average':
            rmsd_values = calculate_rmsd_to_average(u, selection=RMSD_SELECTION)
        elif RMSD_MODE == 'reference':
            rmsd_values = calculate_rmsd_to_reference(u, ref_positions,
                                                      selection=RMSD_SELECTION)
        else:
            raise ValueError(f"RMSD_MODE must be 'average' or 'reference', got {RMSD_MODE!r}")

        sys_results['replicates'].append({
            'name': rep_name,
            'n_frames': n_frames,
            'rmsd_values': rmsd_values,
        })

        print(f"  Mean RMSD: {rmsd_values.mean():.3f} +/- {rmsd_values.std():.3f} A"
              f"  (min {rmsd_values.min():.3f}, max {rmsd_values.max():.3f})")

    # Replicates concatenated end-to-end, matching the H-bond notebook's 'all_rmsd'.
    sys_results['all_rmsd'] = np.concatenate([r['rmsd_values'] for r in sys_results['replicates']])
    rmsd_results[sys_name] = sys_results

print("\nRMSD analysis complete.")

---
## 6. Summary statistics

Statistics are reported over the post-equilibration portion of each replicate,
using the `FRAMES_TO_SKIP` set in the configuration cell — the same value as
`01_hbond_analysis.ipynb`. Set `EQUILIBRATION_NS = 0` to summarise the full
trajectory.

The RMSD traces themselves cover every frame with absolute indices; only the
statistics and the shaded region of the plots respect the skip.

In [ ]:
# FRAMES_TO_SKIP comes from the configuration cell above.
for sys_name, sys_results in rmsd_results.items():
    print("\n" + "=" * 80)
    print(f"RMSD SUMMARY: {sys_name}")
    print(f"Mode: {sys_results['mode']} | Selection: {sys_results['selection']} "
          f"| Frames skipped: {FRAMES_TO_SKIP}")
    print("=" * 80)

    rows = []
    for rep in sys_results['replicates']:
        vals = rep['rmsd_values'][FRAMES_TO_SKIP:]
        rows.append({
            'Replicate': rep['name'],
            'N_Frames': len(vals),
            'Mean_RMSD_A': vals.mean(),
            'SD_RMSD_A': vals.std(),
            'Min_RMSD_A': vals.min(),
            'Max_RMSD_A': vals.max(),
        })

    pooled = np.concatenate([r['rmsd_values'][FRAMES_TO_SKIP:] for r in sys_results['replicates']])
    rows.append({
        'Replicate': 'ALL',
        'N_Frames': len(pooled),
        'Mean_RMSD_A': pooled.mean(),
        'SD_RMSD_A': pooled.std(),
        'Min_RMSD_A': pooled.min(),
        'Max_RMSD_A': pooled.max(),
    })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    csv_name = f"{sys_name}_rmsd_summary.csv"
    summary_df.to_csv(csv_name, index=False)
    print(f"\nSaved to {csv_name}")

## 7. Export the full per-frame traces

In [ ]:
for sys_name, sys_results in rmsd_results.items():
    frames = []
    for rep in sys_results['replicates']:
        for i, val in enumerate(rep['rmsd_values']):
            frames.append({'Replicate': rep['name'], 'Frame': i, 'RMSD_A': val})

    df = pd.DataFrame(frames)
    csv_name = f"{sys_name}_rmsd_timeseries.csv"
    df.to_csv(csv_name, index=False)
    print(f"{sys_name}: {len(df)} rows -> {csv_name}")

## 8. Time-series plots

In [ ]:
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for sys_name, sys_results in rmsd_results.items():
    reps = sys_results['replicates']

    # One panel per replicate, plus a final overlay panel.
    fig, axes = plt.subplots(len(reps) + 1, 1,
                             figsize=(14, 2.2 * (len(reps) + 1)),
                             sharex=True, sharey=True)
    fig.suptitle(f"{sys_name}: RMSD ({sys_results['mode']}) - {sys_results['selection']}",
                 fontsize=14, fontweight='bold')

    for i, rep in enumerate(reps):
        color = COLORS[i % len(COLORS)]
        axes[i].plot(rep['rmsd_values'], color=color, linewidth=0.5, alpha=0.8)
        prod = rep['rmsd_values'][FRAMES_TO_SKIP:]
        axes[i].axhline(prod.mean(), color='black',
                        linestyle='--', linewidth=1,
                        label=f"production mean {prod.mean():.2f} A")
        if FRAMES_TO_SKIP > 0:
            axes[i].axvspan(0, FRAMES_TO_SKIP, color='gray', alpha=0.15)
        axes[i].set_ylabel(f"{rep['name']}\nRMSD (A)")
        axes[i].legend(loc='upper right', fontsize=8)

        # Overlay panel
        axes[-1].plot(rep['rmsd_values'], color=color, linewidth=0.5,
                      alpha=0.7, label=rep['name'])

    axes[-1].set_ylabel('All\nRMSD (A)')
    axes[-1].set_xlabel('Frame')
    axes[-1].legend(loc='upper right', fontsize=8, ncol=len(reps))

    plt.tight_layout()
    plt.savefig(f'{sys_name}_rmsd_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Distribution plots

In [ ]:
for sys_name, sys_results in rmsd_results.items():
    reps = sys_results['replicates']

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, rep in enumerate(reps):
        ax.hist(rep['rmsd_values'][FRAMES_TO_SKIP:], bins=60, histtype='step',
                linewidth=1.5, density=True, color=COLORS[i % len(COLORS)],
                label=rep['name'])

    ax.set_xlabel('RMSD (A)')
    ax.set_ylabel('Probability density')
    ax.set_title(f"{sys_name}: RMSD distribution ({sys_results['mode']}, "
                 f"production frames)")
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{sys_name}_rmsd_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Save the result object and download the files (Colab)

In [ ]:
# ============== SAVE RESULTS ==============
SAVE_NAME = "INSERT_OUTPUT_NAME_rmsd"   # no extension
OUTPUT_FOLDER = "INSERT_OUTPUT_FOLDER"
# ==========================================

save_path = f"{OUTPUT_FOLDER.rstrip('/')}/{SAVE_NAME}.pkl"

with open(save_path, 'wb') as f:
    pickle.dump(rmsd_results, f)

print(f"Saved to: {save_path}")

In [ ]:
import glob

for pattern in ('*_rmsd_summary.csv', '*_rmsd_timeseries.csv',
                '*_rmsd_timeseries.png', '*_rmsd_distribution.png'):
    for f in glob.glob(pattern):
        files.download(f)